In [6]:
import tensorflow as tf
import numpy as np
import h5py
import json
import shutil

# --- 1. Load the Model ---
h5_model_path = 'model.h5'
clean_model_path = 'model_clean.h5'

# Fix the unrecognized keyword 'quantization_config' error
shutil.copy(h5_model_path, clean_model_path)
with h5py.File(clean_model_path, 'r+') as f:
    if 'model_config' in f.attrs:
        model_config = json.loads(f.attrs.get('model_config'))
        for layer in model_config.get('config', {}).get('layers', []):
            if 'config' in layer and 'quantization_config' in layer['config']:
                del layer['config']['quantization_config']
        f.attrs.modify('model_config', json.dumps(model_config).encode('utf-8'))

h5_model = tf.keras.models.load_model(clean_model_path, compile=False)

# --- 2. Understand the Architecture ---
print("=== H5 Model Architecture ===")
h5_model.summary()

# --- 3. Run Inference ---
# Determine the expected input shape from the model
input_shape = h5_model.input_shape
print(f"\nExpected Input Shape: {input_shape}")

# Generate dummy ECG data (replace with an actual normalized beat from MIT-BIH)
# We use [1] for batch size, replacing the 'None' in the input shape
dummy_batch_shape = (1,) + input_shape[1:] 
dummy_float_ecg = np.random.randn(*dummy_batch_shape).astype(np.float32)

# Predict
predictions = h5_model.predict(dummy_float_ecg)
print("\nH5 Model Predictions (Raw Probabilities):")
print(predictions)
print(f"Predicted Class: {np.argmax(predictions, axis=1)[0]}")

=== H5 Model Architecture ===


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 252, 16)        │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 126, 16)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 122, 32)        │         2,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 61, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 59, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,141 (43.52 KB)

 Trainable params: 11,141 (43.52 KB)

 Non-trainable params: 0 (0.00 B)


Expected Input Shape: (None, 256, 1)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step

H5 Model Predictions (Raw Probabilities):
[[9.9909639e-01 6.8860584e-11 1.2185866e-10 1.5087980e-09 9.0361730e-04]]
Predicted Class: 0


In [12]:
# --- 1. Load the Model and Allocate Memory ---
tflite_model_path = 'model_int8.tflite'
interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
interpreter.allocate_tensors()

# --- 2. Understand the Architecture (Tensor Details) ---
print("=== TFLite Model Architecture Summary ===")
print(f"{'Op Name':<20} | {'Input Shape':<20} | {'Output Shape':<20} | {'Params':<10} | {'DType'}")
print("-" * 95)
ops_details = interpreter._get_ops_details()
tensor_details = interpreter.get_tensor_details()

# Identify non-parameter tensors (activations and inputs)
activation_tensors = set([interpreter.get_input_details()[0]['index']])
for op in ops_details:
    for out_idx in op['outputs']:
        activation_tensors.add(out_idx)

total_params = 0

for op in ops_details:
    op_name = op['op_name']
    
    # Skip DELEGATE ops to keep it clean
    if op_name == 'DELEGATE':
        continue
        
    input_shape = str(tensor_details[op['inputs'][0]]['shape']) if len(op['inputs']) > 0 else "None"
    output_shape = str(tensor_details[op['outputs'][0]]['shape']) if len(op['outputs']) > 0 else "None"
    dtype = str(tensor_details[op['outputs'][0]]['dtype'].__name__) if len(op['outputs']) > 0 else "None"
    
    # Calculate parameters for this op (weights + biases)
    op_params = 0
    for in_idx in op['inputs']:
        # If the input is not an activation/global input tensor and is a valid index
        if in_idx != -1 and in_idx not in activation_tensors:
            shape = tensor_details[in_idx]['shape']
            if len(shape) > 0:
                op_params += np.prod(shape)
                
    total_params += op_params
    
    print(f"{op_name:<20} | {input_shape:<20} | {output_shape:<20} | {op_params:<10} | {dtype}")

print("-" * 95)
print(f"Total Parameters: {total_params}")

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

print("\n=== TFLite Input & Output ===")
print(f"Input Name:  {input_details['name']}")
print(f"Input Shape: {input_details['shape']} | Type: {input_details['dtype'].__name__}")
print(f"Quantization (Scale, Zero Point): {input_details['quantization']}")

print(f"\nOutput Shape: {output_details['shape']} | Type: {output_details['dtype'].__name__}")

# --- 3. Run Inference ---
# Generate dummy data matching the required shape
dummy_ecg_shape = input_details['shape']
dummy_float_ecg = np.random.randn(*dummy_ecg_shape).astype(np.float32)

# Check if the model requires Int8 input and quantize if necessary
if input_details['dtype'] == np.int8:
    scale, zero_point = input_details['quantization']
    print("\n[!] Floating dummy data quantized to INT8 for TFLite Inference")
    # Quantization formula: int8_val = (float_val / scale) + zero_point
    dummy_input_data = np.round((dummy_float_ecg / scale) + zero_point)
    dummy_input_data = np.clip(dummy_input_data, -128, 127).astype(np.int8)
else:
    dummy_input_data = dummy_float_ecg

# Set the tensor, run the model, and extract the output
interpreter.set_tensor(input_details['index'], dummy_input_data)
interpreter.invoke()
tflite_predictions = interpreter.get_tensor(output_details['index'])

# If the output is also quantized, dequantize it back to floats to read the probabilities
if output_details['dtype'] == np.int8:
    scale, zero_point = output_details['quantization']
    tflite_predictions = (tflite_predictions.astype(np.float32) - zero_point) * scale

print("\nTFLite Model Predictions (Dequantized to Floats):")
print(tflite_predictions)
print(f"Predicted Class: {np.argmax(tflite_predictions, axis=1)[0]}")

=== TFLite Model Architecture Summary ===
Op Name              | Input Shape          | Output Shape         | Params     | DType
-----------------------------------------------------------------------------------------------
EXPAND_DIMS          | [  1 256   1]        | [  1   1 256   1]    | 0          | int8
CONV_2D              | [  1   1 256   1]    | [  1   1 252  16]    | 96         | int8
RESHAPE              | [  1   1 252  16]    | [  1 252  16]        | 3          | int8
EXPAND_DIMS          | [  1 252  16]        | [  1   1 252  16]    | 0          | int8
MAX_POOL_2D          | [  1   1 252  16]    | [  1   1 126  16]    | 0          | int8
RESHAPE              | [  1   1 126  16]    | [  1 126  16]        | 3          | int8
EXPAND_DIMS          | [  1 126  16]        | [  1   1 126  16]    | 0          | int8
CONV_2D              | [  1   1 126  16]    | [  1   1 122  32]    | 2592       | int8
RESHAPE              | [  1   1 122  32]    | [  1 122  32]        | 3        